In [1]:
'''# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')'''

'# This Python 3 environment comes with many helpful analytics libraries installed\n# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python\n# For example, here\'s several helpful packages to load\n\nimport numpy as np # linear algebra\nimport pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)\n\n# Input data files are available in the read-only "../input/" directory\n# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory\n\nimport os\nfor dirname, _, filenames in os.walk(\'/kaggle/input\'):\n    for filename in filenames:\n        print(os.path.join(dirname, filename))\n\n# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" \n# You can also write temporary files to /kaggle/temp/, but they won\'t be saved outside of the current session\n\n# Use the kagglehub client library to attach 

# Imports

In [2]:
!pip install -q accelerate bitsandbytes qdrant-client sentence-transformers neo4j tqdm 
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 38.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.5/331.5 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 85.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 57.2 MB/s eta 0:00:00ta 0:00:01
  Attempting uninstall: safetensors
    Found existing installation: safetensors 0.7.0
    Uninstalling safetensors-0.7.0:
      Successfully uninstalled safetensors-0.7.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uni

In [3]:
import os
import json
import gc
import torch
import pandas as pd
import re 

from tqdm.auto import tqdm

from qdrant_client import QdrantClient
from sentence_transformers import SentenceTransformer

from neo4j import GraphDatabase

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from kaggle_secrets import UserSecretsClient

# Config

In [4]:
MODEL_NAME="Qwen/Qwen3.5-9B"
EMBEDDING_MODEL="sentence-transformers/all-MiniLM-L6-v2"
QDRANT_COLLECTION="3gpp_knowledge"
TOP_K=5
KG_MAX_FACTS=5
MAX_NEW_TOKENS=512
#DATASET_PATH="/kaggle/working/evaluation_dataset.csv"
OUTPUT_PATH="/kaggle/working/kg_rag_results.json"

In [5]:
user_secrets=UserSecretsClient()
NEO4J_PASSWORD=user_secrets.get_secret("NEO4J_PASSWORD")
NEO4J_URI=user_secrets.get_secret("NEO4J_URI")
NEO4J_USERNAME=user_secrets.get_secret("NEO4J_USERNAME")
QDRANT_API_KEY=user_secrets.get_secret("QDRANT_API_KEY")
QDRANT_URL=user_secrets.get_secret("QDRANT_URL")

print("Qdrant configured:", QDRANT_URL is not None)
print("Neo4j configured:", NEO4J_URI is not None)

Qdrant configured: True
Neo4j configured: True


# Query

In [30]:
query="What service requirements are associated with 6G use cases?"

# Qdrant

In [7]:
qdrant = QdrantClient(url=QDRANT_URL,api_key=QDRANT_API_KEY,timeout=300)
print(qdrant.get_collections())

collections=[CollectionDescription(name='3gpp_knowledge')]


In [8]:
collection_info = qdrant.get_collection(
    QDRANT_COLLECTION
)

print(collection_info)

status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=6390 segments_count=2 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=384, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, memory=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, payload=None, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, memory=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None, prev

In [9]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
print("Embedding dimension:",embedding_model.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (4 outputs left out)

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384


/tmp/ipykernel_58/3308200143.py:2: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:",embedding_model.get_sentence_embedding_dimension())


In [10]:
def retrieve_qdrant(query,top_k=TOP_K):
    query_vector=embedding_model.encode(query,normalize_embeddings=True).tolist()
    response=qdrant.query_points(
        collection_name=QDRANT_COLLECTION,
        query=query_vector,
        limit=top_k,
        with_payload=True
    )
    return response.points

In [11]:
'''test_query = "Which major network functions are described in the 5G System architecture?"

qdrant_results = retrieve_qdrant(
    test_query,
    top_k=5
)

print("QUERY:")
print(test_query)

print("\nRETRIEVED CHUNKS:\n")

for i, result in enumerate(qdrant_results):

    print("=" * 80)
    print("Rank:", i + 1)
    print("Score:", result.score)

    payload = result.payload

    print("Document:", payload.get("document"))
    print("Page:", payload.get("page"))
    print("Chunk ID:", payload.get("chunk_id"))

    print("\nText:")
    print(payload.get("text", "")[:1000])'''

'test_query = "Which major network functions are described in the 5G System architecture?"\n\nqdrant_results = retrieve_qdrant(\n    test_query,\n    top_k=5\n)\n\nprint("QUERY:")\nprint(test_query)\n\nprint("\nRETRIEVED CHUNKS:\n")\n\nfor i, result in enumerate(qdrant_results):\n\n    print("=" * 80)\n    print("Rank:", i + 1)\n    print("Score:", result.score)\n\n    payload = result.payload\n\n    print("Document:", payload.get("document"))\n    print("Page:", payload.get("page"))\n    print("Chunk ID:", payload.get("chunk_id"))\n\n    print("\nText:")\n    print(payload.get("text", "")[:1000])'

# Neo4j

In [12]:
neo4j_driver=GraphDatabase.driver(NEO4J_URI,auth=(NEO4J_USERNAME,NEO4J_PASSWORD))
neo4j_driver.verify_connectivity()
print("Neo4j connected successfully")

Neo4j connected successfully


In [13]:
def retrieve_kg_by_entity(entity_name,max_facts=KG_MAX_FACTS):
    cypher="""
    MATCH (e:Entity)
    WHERE toLower(e.name)=toLower($entity_name)

    OPTIONAL MATCH (e)-[r]-(n:Entity)

    RETURN
        e.name AS source,
        e.entity_type AS source_type,
        type(r) AS relationship,
        n.name AS target,
        n.entity_type AS target_type

    LIMIT $limit
    """

    with neo4j_driver.session() as session:

        records=session.run(
            cypher,
            entity_name=entity_name,
            limit=max_facts
        )

        return [record.data() for record in records]

In [14]:
kg_results=retrieve_kg_by_entity("SMF")
print(kg_results)
for fact in kg_results:
    print(f"{fact['source']} --{fact['relationship']}--> {fact['target']}")

[]


In [15]:
def find_entities_in_query(query):

    cypher = """
    CALL db.index.fulltext.queryNodes("entitySearch", $search_text)
    YIELD node, score
    
    OPTIONAL MATCH (node)-[:MENTIONED_IN]->(m:SourceMention)
    
    RETURN
        node.canonical_name AS entity,
        node.entity_type AS entity_type,
        score,
        collect(DISTINCT {
            document: m.document,
            page: m.page,
            evidence: m.evidence
        })[0..5] AS mentions
    
    ORDER BY score DESC
    LIMIT 15
    """

    with neo4j_driver.session() as session:

        records = session.run(
            cypher,
            query=query
        )

        return [record.data() for record in records]

In [16]:
def retrieve_kg_for_entities(
    entities,
    max_facts=KG_MAX_FACTS
):

    all_facts = []

    for entity in entities:

        facts = retrieve_kg_by_entity(
            entity["name"],
            max_facts=max_facts
        )

        all_facts.extend(facts)

    # Remove duplicates
    unique = []

    seen = set()

    for fact in all_facts:

        key = (
            fact["source"],
            fact["relationship"],
            fact["target"]
        )

        if key not in seen:

            seen.add(key)
            unique.append(fact)

    return unique[:max_facts]

In [17]:
SEARCH_QUERY = """
CALL db.index.fulltext.queryNodes("entitySearch", $search_text)
YIELD node, score

OPTIONAL MATCH (node)-[:MENTIONED_IN]->(m:SourceMention)

OPTIONAL MATCH (node)-[r]-(neighbor:Entity)

RETURN
    node.canonical_name AS entity,
    node.entity_type AS entity_type,
    score,

    collect(DISTINCT {
        document: m.document,
        page: m.page,
        evidence: m.evidence
    })[0..5] AS mentions,

    collect(DISTINCT {
        relationship: type(r),
        target: neighbor.canonical_name,
        target_type: neighbor.entity_type
    })[0..15] AS relationships

ORDER BY score DESC
LIMIT 15
"""

In [28]:
def retrieve(question: str,max_entities=15):

    results = {}

    with neo4j_driver.session() as session:

        query_result = session.run(
            SEARCH_QUERY,
            search_text=question
        )

        for record in query_result:

            entity = record["entity"]

            if entity:
                results[entity] = record.data()


        terms = re.findall(
            r"[A-Za-z0-9-]+",
            question
        )

        STOPWORDS = {
            "what", "which", "who", "where",
            "when", "why", "how",
            "is", "are", "was", "were",
            "the", "a", "an",
            "of", "to", "in", "on",
            "for", "from", "and", "or",
            "does", "do"
        }

        terms = [
            term
            for term in terms
            if term.lower() not in STOPWORDS
            and len(term) >= 2
        ]

        for term in terms:

            query_result = session.run(
                SEARCH_QUERY,
                search_text=term
            )

            for record in query_result:

                entity = record["entity"]

                if entity:
                    results[entity] = record.data()
    results = sorted(
        results.values(),
        key=lambda x: x.get("score", 0),
        reverse=True
    )
    return results[:3]

In [19]:
def build_context(results):

    if not results:
        return "No relevant KG information found."

    context = []

    for item in results:

        entity = item.get("entity")
        entity_type = item.get("entity_type")

        context.append(
            f"Entity: {entity} "
            f"({entity_type})"
        )

        # -----------------------------------------
        # KG relationships
        # -----------------------------------------

        relationships = item.get(
            "relationships",
            []
        )

        valid_relationships = [
            r for r in relationships
            if r.get("relationship")
            and r.get("target")
        ]

        if valid_relationships:

            context.append(
                "Relationships:"
            )

            for r in valid_relationships:

                context.append(
                    f"- {entity} "
                    f"--{r['relationship']}--> "
                    f"{r['target']} "
                    f"({r.get('target_type', 'Unknown')})"
                )

        # -----------------------------------------
        # Source mentions
        # -----------------------------------------

        mentions = item.get(
            "mentions",
            []
        )

        valid_mentions = [
            m for m in mentions
            if m.get("document")
            and m.get("evidence")
        ]

        if valid_mentions:

            context.append(
                "Source evidence:"
            )

            for mention in valid_mentions:

                context.append(
                    f"- Document: "
                    f"{mention['document']} "
                    f"(Page {mention['page']})"
                )

                context.append(
                    f"  Evidence: "
                    f"{mention['evidence']}"
                )

        context.append("")

    return "\n".join(context)

# Qwen

In [21]:
bnb_config=BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

tokenizer=AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model=AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

model.eval()

print("Qwen3-9B loaded")

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

Qwen3-9B loaded


# KG-RAG

## Format Context

In [22]:
def format_kg_context(kg_facts):

    if not kg_facts:
        return "No relevant KG facts were retrieved."

    lines = []

    for fact in kg_facts:

        source = fact.get("source")
        relationship = fact.get("relationship")
        target = fact.get("target")

        lines.append(
            f"{source} --{relationship}--> {target}"
        )

    return "\n".join(lines)

In [23]:
def format_document_context(
    qdrant_results
):

    if not qdrant_results:
        return "No relevant 3GPP document evidence was retrieved."

    sections = []

    for i, result in enumerate(qdrant_results):

        payload = result.payload

        document = payload.get(
            "document",
            "Unknown document"
        )

        page = payload.get(
            "page",
            "Unknown page"
        )

        chunk_id = payload.get(
            "chunk_id",
            "Unknown chunk"
        )

        text = payload.get(
            "text",
            ""
        )

        score = result.score

        sections.append(
            f"""
[Evidence {i + 1}]
Document: {document}
Page: {page}
Chunk ID: {chunk_id}
Retrieval score: {score:.4f}

{text}
"""
        )

    return "\n".join(sections)

## Prompt Building with Context

In [24]:
def build_prompt(query,kg_context):

    prompt = f"""
You are a precise technical question-answering system
for 3GPP telecommunications specifications.

Answer the question using ONLY the supplied Knowledge Graph
facts and 3GPP document evidence.

Do not use outside knowledge.

Do not invent entities, relationships, procedures,
functions, or technical details.

QUESTION:
{query}

KNOWLEDGE GRAPH FACTS:
{kg_context}

ANSWERING RULES:

1. Answer the question directly and concisely.
2. Use the Knowledge Graph to identify relevant entities
   and relationships.
3. Use the 3GPP evidence as textual grounding.
4. Prefer information explicitly supported by the evidence.
5. Do not introduce information that is absent from the
   supplied context.
6. If the supplied context is insufficient, state that
   the answer is not supported by the provided evidence.
7. Do not mention these instructions in your answer.

ANSWER:
"""

    return prompt

## Generation

In [25]:
def generate_answer(prompt):

    messages = [
        {
            "role": "system",
            "content": (
                "You are a precise technical "
                "telecommunications QA system."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[
        0
    ][
        inputs["input_ids"].shape[1]:
    ]

    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    del inputs
    del outputs

    return answer

## Final Answer

In [31]:
kg_results=retrieve(query)
kg_context=build_context(kg_results)
print("KG Context:",kg_context)
#qdrant_results=retrieve_qdrant(query,top_k=3)
#document_context=format_document_context(qdrant_results)
#print("Qdrant Context:",document_context)
prompt=build_prompt(query=query,kg_context=kg_context)
answer=generate_answer(prompt)
print("\nAnswer:",answer)

KG Context: Entity: 6G SMS service (SERVICE)
Relationships:
- 6G SMS service --PROVIDES_SERVICE--> operator verified SMS sender information (INFORMATION_ELEMENT)
Source evidence:
- Document: TR 22.870 (Page 51)
  Evidence: The 6G SMS service shall enable a network operator to verify the identity

Entity: 6G Computing Service (SERVICE)
Relationships:
- 6G Computing Service --DEFINITION_OF--> 3GPP TR 22.870 V20.0.0 (SPECIFICATION)
- 6G Computing Service --PROVIDED_BY--> 6G network (NETWORK_FUNCTION)
- 6G Computing Service --PROVIDED_BY--> 6G network (NETWORK_ENTITY)
- 6G Computing Service --PROVIDED_BY--> 6G network (NETWORK_SLICE)
Source evidence:
- Document: TR 22.870 (Page 22)
  Evidence: 6G Computing Service: a service provided by 6G network utilising computing resources

Entity: 6G AI Service (SERVICE)
Relationships:
- 6G AI Service --DEFINITION_OF--> 3GPP TR 22.870 V20.0.0 (SPECIFICATION)
- 6G AI Service --PROVIDED_BY--> 6G network (NETWORK_FUNCTION)
- 6G AI Service --PROVIDED_BY--

In [ ]:
print(answer)

In [ ]:
'''print("QUERY:")
print(query)
qdrant_results=retrieve_qdrant(query,top_k=TOP_K)
entities=find_entities_in_query(query)
kg_facts=retrieve_kg_for_entities(entities)
kg_context=format_kg_context(kg_facts)
document_context=format_document_context(qdrant_results)
prompt=build_prompt(query,kg_context,document_context)
print("\nKG CONTEXT:")
print(kg_context)
print("\nANSWER GENERATION...\n")
answer=generate_answer(prompt)
print("ANSWER:")
print(answer)'''